<style>
    @import url('https://fonts.googleapis.com/css2?family=Oswald:wght@400;600&display=swap');
    h1.course-title {
        font-family: 'Oswald', sans-serif;
        font-size: 2.4em;
        color: #E7C173;
        letter-spacing: 0.05em;
        border-bottom: 2px solid #E7C173;
        padding-bottom: 0.3em;
        margin-bottom: 0.2em;
    }
    h2.course-subtitle { font-family: 'Oswald', sans-serif; color: #aaa; font-size: 1.2em; }
</style>

<h1 class='course-title'>MACHINE LEARNING IN INDUSTRY</h1>
<h2 class='course-subtitle'>Cardo AI · MSCA Digital Doctoral Network · Day 4</h2>

# Day 4, Block 2 — Experiment Tracking with MLflow

## Table of Contents
1. [Scope and Success Criteria](#1-scope-and-success-criteria)
2. [What is MLflow? The Four Components](#2-what-is-mlflow-the-four-components)
3. [Launch the Tracking Server](#3-launch-the-tracking-server)
4. [Imports and Setup](#4-imports-and-setup)
5. [A First Tracked Run](#5-a-first-tracked-run)
6. [Hyperparameter Sweep — Comparing Runs](#6-hyperparameter-sweep--comparing-runs)
7. [The Model Registry — Staging and Production](#7-the-model-registry--staging-and-production)
8. [Loading a Registered Model](#8-loading-a-registered-model)
9. [Acceptance Checks](#9-acceptance-checks)

---
## 1. Scope and Success Criteria

By the end of this notebook you should be able to:

- **Start** a local MLflow tracking server and navigate its UI
- **Log** hyperparameters, metrics, and a serialised model to an MLflow run
- **Compare** multiple runs in the MLflow UI side by side
- **Register** a model version and transition it through stages (Staging → Production)
- **Load** a registered model and reproduce its predictions

> **Time budget:** this notebook covers Block 2 (≈30 minutes). We move fast — the goal is to *see the workflow*, not memorise every API call.

---
## 2. What is MLflow? The Four Components

MLflow is an open-source platform for the complete ML lifecycle. It has four main components:

| Component | What it does | Used today |
|---|---|---|
| **Tracking** | Logs runs: params, metrics, tags, artifacts | ✅ Blocks 2 & 3 |
| **Projects** | Reproducible run packaging (`MLproject` file) | ✅ `day4/MLproject` |
| **Models** | Model serialisation with "flavors" (sklearn, lgbm, …) | ✅ `predict.py` |
| **Registry** | Model versioning and stage lifecycle | ✅ Block 2 |

**Local vs. production:** Today, MLflow runs entirely on your laptop. In production, you point `MLFLOW_TRACKING_URI` at a shared remote server backed by a cloud storage bucket. The code is *identical* — only the URI changes.

---
## 3. Launch the Tracking Server

Open a **separate terminal**, `cd` to the repo root, and run:

```bash
make -f day4/Makefile mlflow-server
# or directly:
mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db
```

Then open **http://127.0.0.1:5000** in your browser.

> **Why SQLite?** The default file-based store (`mlruns/`) works for a single machine. SQLite gives us a proper database backend that persists cleanly for the Model Registry. In production this would be PostgreSQL or MySQL.

---
## 4. Imports and Setup

In [1]:
# Guard: check the MLflow server is reachable before we proceed
import urllib.request, urllib.error
MLFLOW_URI = "http://127.0.0.1:5000"
try:
    urllib.request.urlopen(f"{MLFLOW_URI}/health", timeout=2)
    print(f"✓ MLflow server is running at {MLFLOW_URI}")
except urllib.error.URLError:
    print(f"⚠  MLflow server not reachable at {MLFLOW_URI}")
    print("   Run in a separate terminal:")
    print("   make -f day4/Makefile mlflow-server")

✓ MLflow server is running at http://127.0.0.1:5000


In [2]:
import sys
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Make day4/ importable
repo_root = Path.cwd().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from day4.src.train import (
    DATA_PATH, SEED, TARGET_BIN_COL,
    build_pipeline, get_feature_columns, load_data, split_data,
)

# ── MLflow configuration ──────────────────────────────────────────────────────
EXPERIMENT_NAME = "adult-income-lgbm"
MODEL_NAME = "adult-income-classifier"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment   : {EXPERIMENT_NAME}")

Tracking URI : http://127.0.0.1:5000
Experiment   : adult-income-lgbm


In [3]:
# ── Load and split data (same conventions as Days 1-2) ────────────────────────
data_path = repo_root / DATA_PATH
df = load_data(data_path)
train_df, val_df, test_df = split_data(df)

numeric_cols, categorical_cols = get_feature_columns(train_df)
feature_cols = numeric_cols + categorical_cols

X_train = train_df[feature_cols]
y_train = train_df[TARGET_BIN_COL]
X_val   = val_df[feature_cols]
y_val   = val_df[TARGET_BIN_COL]

print(f"Train: {len(X_train)} rows | Val: {len(X_val)} rows")
print(f"Features: {len(feature_cols)} ({len(numeric_cols)} numeric, {len(categorical_cols)} categorical)")

Train: 5909 rows | Val: 1488 rows
Features: 19 (12 numeric, 7 categorical)


---
## 5. A First Tracked Run

The core MLflow API is three calls inside a context manager:

```python
with mlflow.start_run():
    mlflow.log_params(...)   # hyperparameters — things you chose
    mlflow.log_metrics(...)  # numbers you measured — AUC, F1, …
    mlflow.sklearn.log_model(pipe, artifact_path="model")  # the serialised model
```

That's it. We are **not** changing the training code — we are just *wrapping* it.

In [4]:
params = {"n_estimators": 300, "learning_rate": 0.1, "max_depth": 5}

with mlflow.start_run(run_name="baseline") as run:
    # 1️⃣ Log what you chose
    mlflow.log_params(params)
    mlflow.log_param("seed", SEED)

    # 2️⃣ Train (same pipeline as Day 2)
    pipe = build_pipeline(numeric_cols, categorical_cols, **params)
    pipe.fit(X_train, y_train)

    # 3️⃣ Measure
    val_proba = pipe.predict_proba(X_val)[:, 1]
    val_pred  = pipe.predict(X_val)
    metrics = {
        "val_auc":      round(roc_auc_score(y_val, val_proba), 4),
        "val_f1":       round(f1_score(y_val, val_pred), 4),
        "val_accuracy": round(accuracy_score(y_val, val_pred), 4),
    }
    mlflow.log_metrics(metrics)

    # 4️⃣ Persist the model
    mlflow.sklearn.log_model(pipe, artifact_path="model")

    baseline_run_id = run.info.run_id

print(f"Run ID  : {baseline_run_id}")
print(f"Metrics : {metrics}")
print(f"\nOpen {MLFLOW_URI} → experiment '{EXPERIMENT_NAME}' → run 'baseline'")

/Users/gennaro.dibrino/projects/ml_industry_course/day4/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gennaro.dibrino/projects/ml_industry_course/day4/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/13 21:53:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 21:53:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13

🏃 View run baseline at: http://127.0.0.1:5000/#/experiments/1/runs/703a92c31d184468a352a85dd1890291
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID  : 703a92c31d184468a352a85dd1890291
Metrics : {'val_auc': 0.9084, 'val_f1': 0.7273, 'val_accuracy': 0.8387}

Open http://127.0.0.1:5000 → experiment 'adult-income-lgbm' → run 'baseline'


**Navigate the MLflow UI now:**
1. Click on the experiment `adult-income-lgbm`
2. Click the run named `baseline`
3. Explore the **Parameters**, **Metrics**, and **Artifacts** tabs
4. Download the logged model and inspect its directory structure

---
## 6. Hyperparameter Sweep — Comparing Runs

Without logging, comparing runs means keeping a spreadsheet (or relying on memory). MLflow makes this systematic.

In [5]:
sweep_configs = [
    {"n_estimators": 100, "learning_rate": 0.10, "max_depth": 3},
    {"n_estimators": 300, "learning_rate": 0.10, "max_depth": 5},  # Day 2 best
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 5},
    {"n_estimators": 500, "learning_rate": 0.05, "max_depth": 7},
]

sweep_results = []
for cfg in sweep_configs:
    run_label = f"n{cfg['n_estimators']}_lr{cfg['learning_rate']}_d{cfg['max_depth']}"
    with mlflow.start_run(run_name=run_label) as run:
        mlflow.log_params(cfg)
        pipe = build_pipeline(numeric_cols, categorical_cols, **cfg)
        pipe.fit(X_train, y_train)
        val_proba = pipe.predict_proba(X_val)[:, 1]
        auc = round(roc_auc_score(y_val, val_proba), 4)
        mlflow.log_metric("val_auc", auc)
        mlflow.sklearn.log_model(pipe, artifact_path="model")
        sweep_results.append({**cfg, "val_auc": auc, "run_id": run.info.run_id})
        print(f"  {run_label:45s}  val_auc={auc:.4f}")

sweep_df = pd.DataFrame(sweep_results).sort_values("val_auc", ascending=False)
display(sweep_df)
print(f"\nBest run: {sweep_df.iloc[0]['run_id']}  (val_auc={sweep_df.iloc[0]['val_auc']:.4f})")

/Users/gennaro.dibrino/projects/ml_industry_course/day4/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/13 21:55:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 21:55:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 21:55:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


  n100_lr0.1_d3                                  val_auc=0.9109
🏃 View run n100_lr0.1_d3 at: http://127.0.0.1:5000/#/experiments/1/runs/aca641584c6e4235ac9490492176f6b6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/gennaro.dibrino/projects/ml_industry_course/day4/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/13 21:55:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 21:55:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 21:55:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


  n300_lr0.1_d5                                  val_auc=0.9084
🏃 View run n300_lr0.1_d5 at: http://127.0.0.1:5000/#/experiments/1/runs/9e49baac9cfa4cd48362a4bcb96b6342
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/gennaro.dibrino/projects/ml_industry_course/day4/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/13 21:55:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 21:55:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 21:55:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


  n300_lr0.05_d5                                 val_auc=0.9118
🏃 View run n300_lr0.05_d5 at: http://127.0.0.1:5000/#/experiments/1/runs/6880e06ba3a44772b5fe48b8894f10c6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/gennaro.dibrino/projects/ml_industry_course/day4/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/13 21:55:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 21:55:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 21:55:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


  n500_lr0.05_d7                                 val_auc=0.9055
🏃 View run n500_lr0.05_d7 at: http://127.0.0.1:5000/#/experiments/1/runs/fe55b4c07c83445baa1f9806b0519181
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,n_estimators,learning_rate,max_depth,val_auc,run_id
2,300,0.05,5,0.9118,6880e06ba3a44772b5fe48b8894f10c6
0,100,0.10,3,0.9109,aca641584c6e4235ac9490492176f6b6
1,300,0.10,5,0.9084,9e49baac9cfa4cd48362a4bcb96b6342
3,500,0.05,7,0.9055,fe55b4c07c83445baa1f9806b0519181



Best run: 6880e06ba3a44772b5fe48b8894f10c6  (val_auc=0.9118)


**In the MLflow UI:**
- Select all four runs and click **Compare**
- Use the parallel coordinates plot to see which hyperparameter drives AUC the most
- This is the workflow a team uses before promoting a model to staging

---
## 7. The Model Registry — Staging and Production

The **Tracking** server stores experiment records. The **Registry** is a separate layer that manages the *deployment lifecycle* of a model version:

```
None (just logged) → Staging (candidate) → Production (live) → Archived
```

In a financial services context, moving a model to Production typically requires a sign-off from a model risk team. MLflow captures that paper trail: who promoted it, when, and from which run.

In [6]:
best_run_id = sweep_df.iloc[0]["run_id"]
model_uri   = f"runs:/{best_run_id}/model"

# Register the best run's model
result = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
version = result.version
print(f"Registered version: {version}")

Successfully registered model 'adult-income-classifier'.
2026/03/13 21:56:14 WARNING mlflow.tracking._model_registry.fluent: Run with id 6880e06ba3a44772b5fe48b8894f10c6 has no artifacts at artifact path 'model', registering model based on models:/m-53c3fe92462041d29f40ff77a36422ce instead
2026/03/13 21:56:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: adult-income-classifier, version 1
Created version '1' of model 'adult-income-classifier'.


Registered version: 1


In [7]:
import time
client = mlflow.tracking.MlflowClient()

# Transition through stages
client.transition_model_version_stage(
    name=MODEL_NAME, version=version, stage="Staging",
)
print(f"Version {version} → Staging")

time.sleep(1)  # give the server a moment

client.transition_model_version_stage(
    name=MODEL_NAME, version=version, stage="Production",
)
print(f"Version {version} → Production")
print(f"\nOpen {MLFLOW_URI}/#/models/{MODEL_NAME} to inspect the Registry.")

/var/folders/93/ds91tyqj3yq9343kfg7rpdxc0000gn/T/ipykernel_43330/2862676094.py:5: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Version 1 → Staging
Version 1 → Production

Open http://127.0.0.1:5000/#/models/adult-income-classifier to inspect the Registry.


/var/folders/93/ds91tyqj3yq9343kfg7rpdxc0000gn/T/ipykernel_43330/2862676094.py:12: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


---
## 8. Loading a Registered Model

Two load patterns:
- `models:/<name>/<stage>` — always loads whatever is in Production (good for serving)
- `runs:/<run_id>/model` — loads a specific run's artifact (good for reproducibility)

In [8]:
# Pattern 1: load from Registry by stage
prod_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/Production")

# Pattern 2: load from a specific run
run_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")

# Both should produce identical predictions
sample = X_val.head(5)
pred_prod = prod_model.predict_proba(sample)[:, 1]
pred_run  = run_model.predict_proba(sample)[:, 1]

assert np.allclose(pred_prod, pred_run), "Models are NOT identical!"
print("✓ Both models produce identical predictions.")
print(pd.DataFrame({"P(>50K) from Registry": pred_prod, "P(>50K) from Run": pred_run}))

✓ Both models produce identical predictions.
   P(>50K) from Registry  P(>50K) from Run
0               0.095348          0.095348
1               0.306575          0.306575
2               0.413084          0.413084
3               0.532430          0.532430
4               0.749068          0.749068


/Users/gennaro.dibrino/projects/ml_industry_course/day4/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gennaro.dibrino/projects/ml_industry_course/day4/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


---
## 9. Acceptance Checks

In [9]:
# ── Run all acceptance checks ─────────────────────────────────────────────────
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
assert exp is not None, f"Experiment '{EXPERIMENT_NAME}' not found"

runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
assert len(runs) >= 4, f"Expected ≥4 runs, found {len(runs)}"

assert runs["metrics.val_auc"].notna().all(), "Some runs are missing val_auc"
assert (runs["metrics.val_auc"] > 0.5).all(), "val_auc below chance level — something is wrong"

versions = client.get_latest_versions(MODEL_NAME, stages=["Production"])
assert len(versions) >= 1, f"No model version in Production"

print("✓ Experiment exists")
print(f"✓ {len(runs)} runs logged")
print(f"✓ All runs have val_auc  (best: {runs['metrics.val_auc'].max():.4f})")
print(f"✓ Model '{MODEL_NAME}' has a Production version (v{versions[0].version})")
print("\nAll acceptance checks passed. 🎉")

✓ Experiment exists
✓ 5 runs logged
✓ All runs have val_auc  (best: 0.9118)
✓ Model 'adult-income-classifier' has a Production version (v1)

All acceptance checks passed. 🎉


/var/folders/93/ds91tyqj3yq9343kfg7rpdxc0000gn/T/ipykernel_43330/3956740594.py:11: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  versions = client.get_latest_versions(MODEL_NAME, stages=["Production"])
